# Accelerating Option Pricing with Neural Networks
## Reproducible Research Notebook: Black-Scholes to Heston

**Research objective.** Test whether a neural-network surrogate can reduce the computational cost of repeated option valuation, while explicitly accounting for training cost, approximation error, uncertainty across model seeds, and the implementation-specific cost of Monte Carlo.

The experiments are deliberately structured so that Black-Scholes is a controlled benchmark and Heston is the main computational use case. The Heston Monte Carlo reference engine is compiled with Numba and uses antithetic paths for a stronger computational baseline.

## 1. Research design

We answer three questions:

1. Can a feed-forward neural network accurately learn the Black-Scholes pricing surface?
2. Does the computational case for a neural-network surrogate become stronger for Heston, where reference pricing requires numerical simulation?
3. After accounting for training cost and benchmark variability, how many repeated pricing queries are required before the surrogate becomes computationally attractive?

The key quantity is **amortized cost**. The study also reports uncertainty across multiple neural-network seeds and repeated timing runs, and uses a compiled antithetic Monte Carlo engine rather than a Python-level simulation loop.

In [ ]:
import os, time, platform, math, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as si
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import torch
import torch.nn as nn
import torch.optim as optim
from numba import njit, prange

SEED = 20260831
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("CPU:", platform.processor() or platform.machine())
print("Torch threads:", torch.get_num_threads())

from scipy.stats import t as student_t

def mean_ci95(values):
    values=np.asarray(values,dtype=float)
    n=len(values)
    mean=float(np.mean(values))
    if n < 2:
        return mean, np.nan, np.nan
    se=float(np.std(values,ddof=1)/np.sqrt(n))
    margin=float(student_t.ppf(0.975,n-1)*se)
    return mean, mean-margin, mean+margin

def bootstrap_ci(values, stat_fn=np.median, n_boot=5000, seed=SEED):
    values=np.asarray(values,dtype=float)
    rng=np.random.default_rng(seed)
    idx=rng.integers(0,len(values),size=(n_boot,len(values)))
    stats=np.apply_along_axis(stat_fn,1,values[idx])
    return float(stat_fn(values)), float(np.quantile(stats,0.025)), float(np.quantile(stats,0.975))

def bootstrap_speedup_ci(mc_times, nn_times, n_boot=5000, seed=SEED+99):
    mc_times=np.asarray(mc_times,dtype=float); nn_times=np.asarray(nn_times,dtype=float)
    rng=np.random.default_rng(seed)
    mc_idx=rng.integers(0,len(mc_times),size=(n_boot,len(mc_times)))
    nn_idx=rng.integers(0,len(nn_times),size=(n_boot,len(nn_times)))
    ratios=np.median(mc_times[mc_idx],axis=1)/np.median(nn_times[nn_idx],axis=1)
    return float(np.median(mc_times)/np.median(nn_times)), float(np.quantile(ratios,0.025)), float(np.quantile(ratios,0.975))


## 2. Black-Scholes benchmark

In [ ]:
def black_scholes_call(S, K, T, r, sigma):
    S = np.asarray(S, dtype=float)
    K = np.asarray(K, dtype=float)
    T = np.asarray(T, dtype=float)
    r = np.asarray(r, dtype=float)
    sigma = np.asarray(sigma, dtype=float)
    sqrtT = np.sqrt(T)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * sqrtT)
    d2 = d1 - sigma * sqrtT
    return S * si.norm.cdf(d1) - K * np.exp(-r * T) * si.norm.cdf(d2)

benchmark = dict(S=100.0, K=100.0, T=1.0, r=0.05, sigma=0.20)
bs_price = float(black_scholes_call(**benchmark))
print(f"Black-Scholes benchmark price: {bs_price:.6f}")

## 3. Black-Scholes neural surrogate

In [ ]:
rng = np.random.default_rng(SEED)
n_bs = 50000
S = rng.uniform(80, 120, n_bs)
K = rng.uniform(80, 120, n_bs)
T = rng.uniform(0.05, 2.0, n_bs)
r = rng.uniform(0.005, 0.08, n_bs)
sigma = rng.uniform(0.10, 0.60, n_bs)
X_bs = np.column_stack([S, K, T, r, sigma])
y_bs = black_scholes_call(S, K, T, r, sigma)

Xtr, Xte, ytr, yte = train_test_split(X_bs, y_bs, test_size=0.20, random_state=SEED)
sc_bs = StandardScaler().fit(Xtr)
Xtr_t = torch.tensor(sc_bs.transform(Xtr), dtype=torch.float32)
Xte_t = torch.tensor(sc_bs.transform(Xte), dtype=torch.float32)
ytr_t = torch.tensor(ytr[:,None], dtype=torch.float32)

class MLP(nn.Module):
    def __init__(self, d_in, hidden=(64,64,32)):
        super().__init__()
        layers=[]
        d=d_in
        for h in hidden:
            layers += [nn.Linear(d,h), nn.ReLU()]
            d=h
        layers += [nn.Linear(d,1)]
        self.net=nn.Sequential(*layers)
    def forward(self,x): return self.net(x)

def train_model(X, y, d_in, epochs=500, lr=1e-3, seed=SEED):
    torch.manual_seed(seed)
    m=MLP(d_in)
    opt=optim.Adam(m.parameters(), lr=lr)
    loss_fn=nn.MSELoss()
    start=time.perf_counter()
    losses=[]
    for epoch in range(epochs):
        opt.zero_grad()
        pred=m(X)
        loss=loss_fn(pred,y)
        loss.backward()
        opt.step()
        if (epoch+1)%100==0: losses.append(float(loss.item()))
    return m, time.perf_counter()-start, losses

bs_model, bs_train_time, bs_losses = train_model(Xtr_t, ytr_t, 5, epochs=500)
bs_model.eval()
with torch.no_grad():
    bs_pred=bs_model(Xte_t).numpy().ravel()
bs_mae=mean_absolute_error(yte,bs_pred)
bs_rmse=mean_squared_error(yte,bs_pred)**0.5
bs_norm_mae=float(bs_mae/np.mean(np.abs(yte)))
bs_rel_mask=yte>=1.0
bs_rel_mae_ge1=float(np.mean(np.abs(bs_pred[bs_rel_mask]-yte[bs_rel_mask])/yte[bs_rel_mask]))
print(f"BS NN training time: {bs_train_time:.4f} s")
print(f"BS NN MAE: {bs_mae:.6f}")
print(f"BS NN RMSE: {bs_rmse:.6f}")
print(f"BS NN normalized MAE (MAE / mean price): {bs_norm_mae:.4%}")
print(f"BS NN mean relative absolute error (price >= 1): {bs_rel_mae_ge1:.4%}")

In [ ]:
plt.figure(figsize=(7,5))
plt.scatter(yte, bs_pred, s=6, alpha=0.25, label="NN predictions")
lo=min(yte.min(),bs_pred.min()); hi=max(yte.max(),bs_pred.max())
plt.plot([lo,hi],[lo,hi], linestyle="--", label="Perfect fit")
plt.xlabel("Black-Scholes price")
plt.ylabel("NN price")
plt.title("Black-Scholes surrogate: held-out test set")
plt.legend(); plt.tight_layout(); plt.show()

## 4. Fair repeated-pricing benchmark under Black-Scholes

In [ ]:
@njit(fastmath=True)
def mc_bs_single(S,K,T,r,sigma,n_paths=5000,steps=50,seed=1):
    np.random.seed(seed)
    dt=T/steps
    sqdt=np.sqrt(dt)
    total=0.0
    for _ in range(n_paths):
        x=S
        for _ in range(steps):
            z=np.random.randn()
            x *= np.exp((r-0.5*sigma*sigma)*dt + sigma*sqdt*z)
        total += max(x-K,0.0)
    return np.exp(-r*T)*total/n_paths

# Warm-up compilation, then repeat the benchmark to quantify timing variability.
_ = mc_bs_single(100,100,1,.05,.2,100,10,SEED)
bs_mc_times=[]
bs_mc_prices=[]
for rep in range(5):
    start=time.perf_counter()
    p=float(mc_bs_single(100,100,1,.05,.2,5000,50,SEED+rep))
    bs_mc_times.append(time.perf_counter()-start)
    bs_mc_prices.append(p)
mc_price=float(np.mean(bs_mc_prices))
mc_runtime=float(np.median(bs_mc_times))
mc_runtime_mean,mc_runtime_lo,mc_runtime_hi=mean_ci95(bs_mc_times)
print(f"Monte Carlo mean price over 5 runs: {mc_price:.6f}")
print(f"Absolute error vs BS: {abs(mc_price-bs_price):.6f}")
print(f"Monte Carlo runtime: {mc_runtime:.6f} s (median; 95% CI for mean: {mc_runtime_lo:.6f}-{mc_runtime_hi:.6f} s)")


## 5. Heston Monte Carlo reference engine

In [ ]:
# Compiled antithetic Monte Carlo. Each option receives a deterministic seed (base seed + row index).
@njit(parallel=True, fastmath=True)
def heston_mc_batch(params, n_paths, steps, base_seed):
    n=params.shape[0]
    prices=np.empty(n)
    for i in prange(n):
        S0,V0,K,T,r,kappa,theta,xi,rho=params[i]
        dt=T/steps
        sqdt=np.sqrt(dt)
        sqrt_corr=np.sqrt(max(1.0-rho*rho,0.0))
        half=n_paths//2
        total=0.0
        np.random.seed(base_seed+i)
        for p in range(half):
            S1=S0; V1=V0
            S2=S0; V2=V0
            for _ in range(steps):
                z1=np.random.randn(); z2=np.random.randn()
                zv=rho*z1 + sqrt_corr*z2
                vp=max(V1,0.0)
                S1 *= np.exp((r-0.5*vp)*dt + np.sqrt(vp*dt)*z1)
                V1=max(vp + kappa*(theta-vp)*dt + xi*np.sqrt(vp*dt)*zv,0.0)

                z1a=-z1; z2a=-z2
                zva=rho*z1a + sqrt_corr*z2a
                vp2=max(V2,0.0)
                S2 *= np.exp((r-0.5*vp2)*dt + np.sqrt(vp2*dt)*z1a)
                V2=max(vp2 + kappa*(theta-vp2)*dt + xi*np.sqrt(vp2*dt)*zva,0.0)
            total += 0.5*(max(S1-K,0.0)+max(S2-K,0.0))
        prices[i]=np.exp(-r*T)*total/half
    return prices

## 6. Heston synthetic dataset

The Heston surrogate is trained on 8,000 synthetic parameter vectors. To reduce Monte Carlo label noise, the training targets use **5,000 antithetic paths and 100 time steps per option**. The timing benchmark later in the notebook remains fixed at 3,000 paths and 100 steps so that the computational comparison is not changed by the accuracy experiment.


In [ ]:
rng=np.random.default_rng(SEED+1)
n_heston=8000
hS=rng.uniform(80,120,n_heston)
hV=rng.uniform(0.01,0.12,n_heston)
hK=rng.uniform(80,120,n_heston)
hT=rng.uniform(0.10,3.0,n_heston)
hr=rng.uniform(0.005,0.08,n_heston)
hk=rng.uniform(0.5,5.0,n_heston)
htheta=rng.uniform(0.01,0.12,n_heston)
hxi=rng.uniform(0.10,0.70,n_heston)
hrho=rng.uniform(-0.95,-0.05,n_heston)
H=np.column_stack([hS,hV,hK,hT,hr,hk,htheta,hxi,hrho])

# Higher-accuracy training labels: 5,000 antithetic MC paths per option.
HESTON_MC_PATHS_TRAIN = 5000
HESTON_MC_STEPS_TRAIN = 100

start=time.perf_counter()
hy=heston_mc_batch(
    H,
    n_paths=HESTON_MC_PATHS_TRAIN,
    steps=HESTON_MC_STEPS_TRAIN,
    base_seed=SEED+100
)
h_generation=time.perf_counter()-start
print(f"Generated {n_heston:,} Heston labels in {h_generation:.2f} s")
print(f"MC paths per training label: {HESTON_MC_PATHS_TRAIN:,}")
print(f"MC time steps per training label: {HESTON_MC_STEPS_TRAIN}")
print(f"Price range: {hy.min():.4f} to {hy.max():.4f}")


## 7. Heston neural surrogate and accuracy improvements

The Heston model uses four controlled changes intended to improve price accuracy without adding unnecessary model complexity: **(1)** up to 1,000 training epochs with early stopping, **(2)** a modestly larger feed-forward network, **(3)** Huber loss for robustness to large residuals, and **(4)** higher-accuracy 5,000-path Monte Carlo training labels. Three independent random seeds are used to quantify model variability.


In [ ]:
# ============================================================
# 7. Improved Heston neural surrogate
#    1. Up to 1,000 epochs + early stopping
#    2. Larger architecture: 9 -> 128 -> 128 -> 64 -> 32 -> 1
#    3. Huber loss
#    4. 3 independent seeds
# ============================================================

import copy

# Train / validation / test split
HXtr,HXte,hytr,hyte=train_test_split(
    H,hy,test_size=0.20,random_state=SEED
)
HX_train,HX_val,hy_train,hy_val=train_test_split(
    HXtr,hytr,test_size=0.20,random_state=SEED+10
)

# Fit scaling on training data only
sc_h=StandardScaler().fit(HX_train)
HX_train_t=torch.tensor(sc_h.transform(HX_train),dtype=torch.float32)
HX_val_t=torch.tensor(sc_h.transform(HX_val),dtype=torch.float32)
HXte_t=torch.tensor(sc_h.transform(HXte),dtype=torch.float32)
hy_train_t=torch.tensor(hy_train[:,None],dtype=torch.float32)
hy_val_t=torch.tensor(hy_val[:,None],dtype=torch.float32)

class HestonMLP(nn.Module):
    def __init__(self,d_in=9):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(d_in,128),
            nn.ReLU(),
            nn.Linear(128,128),
            nn.ReLU(),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Linear(64,32),
            nn.ReLU(),
            nn.Linear(32,1)
        )
    def forward(self,x):
        return self.net(x)

def train_heston_model(
    X_train,y_train,X_val,y_val,
    epochs=1000,lr=1e-3,patience=50,delta=1.0,seed=SEED
):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model=HestonMLP(d_in=9)
    optimizer=optim.Adam(model.parameters(),lr=lr)
    loss_fn=nn.HuberLoss(delta=delta)
    best_val_loss=np.inf
    best_state=None
    epochs_without_improvement=0
    train_losses=[]
    val_losses=[]
    start=time.perf_counter()

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        train_pred=model(X_train)
        train_loss=loss_fn(train_pred,y_train)
        train_loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_pred=model(X_val)
            val_loss=loss_fn(val_pred,y_val)

        tr=float(train_loss.item())
        va=float(val_loss.item())
        train_losses.append(tr)
        val_losses.append(va)

        if va < best_val_loss - 1e-6:
            best_val_loss=va
            best_state=copy.deepcopy(model.state_dict())
            epochs_without_improvement=0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    train_time=time.perf_counter()-start
    if best_state is not None:
        model.load_state_dict(best_state)

    return model,train_time,train_losses,val_losses,epoch+1,best_val_loss

seed_results=[]
for seed in [SEED,SEED+1,SEED+2]:
    (model,t,train_losses,val_losses,epochs_used,best_val_loss)=train_heston_model(
        HX_train_t,hy_train_t,HX_val_t,hy_val_t,
        epochs=1000,lr=1e-3,patience=50,delta=1.0,seed=seed
    )
    model.eval()
    with torch.no_grad():
        pred=model(HXte_t).numpy().ravel()
    mae=float(mean_absolute_error(hyte,pred))
    rmse=float(mean_squared_error(hyte,pred)**0.5)
    normalized_mae=float(mae/np.mean(np.abs(hyte)))
    rel_mask=hyte>=1.0
    rel_mae_ge1=float(np.mean(np.abs(pred[rel_mask]-hyte[rel_mask])/hyte[rel_mask]))
    seed_results.append({
        'seed':seed,
        'train_time_s':t,
        'epochs_used':epochs_used,
        'best_val_loss':best_val_loss,
        'MAE':mae,
        'RMSE':rmse,
        'normalized_MAE':normalized_mae,
        'relative_MAE_price_ge_1':rel_mae_ge1,
        'model':model,
        'pred':pred,
        'train_losses':train_losses,
        'val_losses':val_losses
    })

seed_table=pd.DataFrame([
    {k:v for k,v in r.items() if k not in ['model','pred','train_losses','val_losses']}
    for r in seed_results
])
print(seed_table.to_string(index=False))
print('\n95% confidence intervals across three model seeds')
for metric in ['MAE','RMSE','normalized_MAE','train_time_s']:
    mean_v,lo,hi=mean_ci95(seed_table[metric].values)
    print(f'{metric}: mean={mean_v:.6f}, 95% CI=({lo:.6f}, {hi:.6f})')

best_idx=int(seed_table['MAE'].idxmin())
heston_model=seed_results[best_idx]['model']
heston_pred=seed_results[best_idx]['pred']
print('\nSelected seed:',seed_results[best_idx]['seed'])
print('Selected model epochs:',seed_results[best_idx]['epochs_used'])


In [ ]:
best_result=seed_results[best_idx]
plt.figure(figsize=(7,5))
plt.plot(best_result['train_losses'],label='Training loss')
plt.plot(best_result['val_losses'],label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Huber loss')
plt.title('Heston neural-network training')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7,5))
plt.scatter(hyte,heston_pred,s=7,alpha=0.28,label='NN predictions')
lo=min(hyte.min(),heston_pred.min())
hi=max(hyte.max(),heston_pred.max())
plt.plot([lo,hi],[lo,hi],'--',label='Perfect fit')
plt.xlabel('Heston Monte Carlo price')
plt.ylabel('Neural-network price')
plt.title('Heston neural-network approximation')
plt.legend()
plt.tight_layout()
plt.show()


## 8. Error analysis by moneyness and maturity

In [ ]:
err=np.abs(heston_pred-hyte)
norm_err=err/np.mean(np.abs(hyte))
rel_mask=hyte>=1.0
rel=np.full_like(err,np.nan,dtype=float)
rel[rel_mask]=err[rel_mask]/hyte[rel_mask]
report=pd.DataFrame({
    'S0':HXte[:,0],'V0':HXte[:,1],'K':HXte[:,2],'T':HXte[:,3],
    'abs_error':err,'normalized_error':norm_err,'rel_error_price_ge_1':rel
})
report['moneyness']=report['S0']/report['K']
report['moneyness_bucket']=pd.cut(report['moneyness'],[-np.inf,0.95,1.05,np.inf],labels=['OTM','Near ATM','ITM'])
report['maturity_bucket']=pd.cut(report['T'],[0,0.5,1.0,2.0,np.inf],labels=['0-0.5y','0.5-1y','1-2y','2y+'])
print('Error by moneyness (relative error excludes options with reference price < 1.0)')
print(report.groupby('moneyness_bucket',observed=False)[['abs_error','normalized_error','rel_error_price_ge_1']].mean().to_string())
print('
Error by maturity (relative error excludes options with reference price < 1.0)')
print(report.groupby('maturity_bucket',observed=False)[['abs_error','normalized_error','rel_error_price_ge_1']].mean().to_string())


In [ ]:
fig,ax=plt.subplots(figsize=(7,5))
for label,g in report.groupby('moneyness_bucket',observed=False):
    ax.scatter(g['moneyness'],g['abs_error'],s=7,alpha=0.3,label=str(label))
ax.set_xlabel('Moneyness S/K'); ax.set_ylabel('Absolute pricing error'); ax.set_title('Heston error versus moneyness'); ax.legend(); fig.tight_layout(); plt.show()

## 9. Stronger benchmark: compiled Heston Monte Carlo versus NN inference

The computational benchmark uses the same held-out parameter vectors for both methods and retains **3,000 antithetic Monte Carlo paths and 100 time steps**. Compiled Monte Carlo and PyTorch inference are warmed up before repeated timing. This benchmark is intentionally separate from the higher-accuracy 5,000-path training-label experiment.


In [ ]:
# Stronger benchmark: compiled, antithetic Heston Monte Carlo versus NN inference.
# Both methods use exactly the same held-out parameter vectors and the same 3,000-path/100-step settings.
# Warm up compiled MC and PyTorch before timing.
_ = heston_mc_batch(HXte[:4], n_paths=100, steps=20, base_seed=999)
with torch.no_grad(): _=heston_model(HXte_t[:4])

def repeated_times(fn, repeats, seed_offset=0):
    times=[]
    for rep in range(repeats):
        start=time.perf_counter()
        fn(seed_offset+rep)
        times.append(time.perf_counter()-start)
    return np.asarray(times,dtype=float)

mc_times=repeated_times(
    lambda rep: heston_mc_batch(HXte, n_paths=3000, steps=100, base_seed=2000+rep),
    repeats=5
)

nn_times=[]
for _ in range(20):
    start=time.perf_counter()
    with torch.no_grad(): _=heston_model(HXte_t)
    nn_times.append(time.perf_counter()-start)
nn_times=np.asarray(nn_times,dtype=float)

mc_h_time=float(np.median(mc_times))
nn_h_time=float(np.median(nn_times))
mc_h_mean,mc_h_lo,mc_h_hi=mean_ci95(mc_times)
nn_h_mean,nn_h_lo,nn_h_hi=mean_ci95(nn_times)
ratio,ratio_lo,ratio_hi=bootstrap_speedup_ci(mc_times,nn_times)

print(f"Heston MC: {mc_h_time:.6f} s (median over 5 runs; mean 95% CI {mc_h_lo:.6f}-{mc_h_hi:.6f} s)")
print(f"Heston NN: {nn_h_time:.6f} s (median over 20 runs; mean 95% CI {nn_h_lo:.6f}-{nn_h_hi:.6f} s)")
print(f"Speed-up: {ratio:.1f}x (bootstrap 95% CI {ratio_lo:.1f}x-{ratio_hi:.1f}x)")
print(f"MC time / option: {mc_h_time/len(HXte):.8f} s")
print(f"NN time / option: {nn_h_time/len(HXte):.8f} s")
print(f"MC timing runs: {mc_times}")
print(f"NN timing median/IQR: {np.median(nn_times):.8f} / {np.quantile(nn_times,0.75)-np.quantile(nn_times,0.25):.8f} s")


## 10. Amortized economics

In [ ]:
train_cost=seed_results[best_idx]['train_time_s']
mc_per_option=mc_h_time/len(HXte)
nn_per_option=nn_h_time/len(HXte)
breakeven=train_cost/(mc_per_option-nn_per_option) if mc_per_option>nn_per_option else np.inf
print(f"One-time training cost: {train_cost:.6f} s")
print(f"MC per option: {mc_per_option:.9f} s")
print(f"NN per option: {nn_per_option:.9f} s")
print(f"Amortized break-even: {breakeven:,.0f} options")
print(f"Timing uncertainty: MC mean 95% CI {mc_h_lo:.6f}-{mc_h_hi:.6f} s; NN mean 95% CI {nn_h_lo:.6f}-{nn_h_hi:.6f} s")
print(f"Speed-up uncertainty: bootstrap 95% CI {ratio_lo:.1f}x-{ratio_hi:.1f}x")

N=np.logspace(1,7,300)
tnn=train_cost+N*nn_per_option
tmc=N*mc_per_option
plt.figure(figsize=(7,5))
plt.loglog(N,tnn,label='NN: training + inference')
plt.loglog(N,tmc,label='Heston Monte Carlo')
if np.isfinite(breakeven):
    plt.axvline(breakeven,linestyle='--',label=f'Break-even = {breakeven:,.0f}')
plt.xlabel('Number of pricing queries'); plt.ylabel('Cumulative runtime (s)')
plt.title('Amortized computational cost'); plt.legend(); plt.tight_layout(); plt.show()

## 11. High-accuracy reference check on a small Heston subset

In [ ]:
# Re-price 100 held-out points with 20,000 antithetic paths and compare the selected network.
subset=HXte[:100]
start=time.perf_counter()
hy_hi=heston_mc_batch(subset,n_paths=20000,steps=200,base_seed=5000)
hi_time=time.perf_counter()-start
with torch.no_grad(): hi_pred=heston_model(torch.tensor(sc_h.transform(subset),dtype=torch.float32)).numpy().ravel()
hi_mae=mean_absolute_error(hy_hi,hi_pred)
hi_rmse=mean_squared_error(hy_hi,hi_pred)**0.5
print(f"High-accuracy MC revaluation time (100 options): {hi_time:.3f} s")
print(f"NN MAE against high-accuracy MC: {hi_mae:.6f}")
print(f"NN RMSE against high-accuracy MC: {hi_rmse:.6f}")

## 12. Interpretation and publication notes

The intended conclusion is conditional rather than universal:

- **Black-Scholes:** the analytical formula remains the appropriate benchmark; a neural network adds approximation error and training cost without a compelling computational reason to replace the formula.
- **Heston:** the surrogate argument is materially stronger because the reference pricing operator is numerical. The relevant decision is whether the reduction in repeated valuation cost is large enough to justify training while maintaining acceptable pricing error.
- **Accuracy reporting:** raw mean relative absolute error can be dominated by very small option prices. The study therefore uses MAE, RMSE and normalized MAE as the primary metrics, while any relative-error diagnostic excludes reference prices below 1.0 monetary unit and is clearly labeled.
- **Robustness:** results are reported across three neural-network seeds, and timing results are repeated after warm-up. The Heston speed comparison uses the compiled antithetic Monte Carlo engine described above, so the reported speed-up is an implementation-specific benchmark rather than a universal claim.

### Reproducibility checklist

- Fixed seeds for synthetic data and model initialization.
- Explicit parameter ranges in code.
- Compiled, antithetic Heston Monte Carlo for both reference labels and the runtime benchmark.
- Repeated timing after warm-up, with confidence intervals and bootstrap uncertainty for the speed-up.
- Independent held-out test set.
- Multi-seed neural-network robustness check.
- Small high-accuracy Monte Carlo revaluation as a robustness check.


## 13. Save machine-readable benchmark results

The results are written to a local `publication_outputs/` directory so the notebook remains portable across Windows, macOS and Linux.

In [ ]:
results={
    'black_scholes_price':bs_price,
    'black_scholes_nn_MAE':bs_mae,
    'black_scholes_nn_RMSE':bs_rmse,
    'black_scholes_nn_normalized_MAE':bs_norm_mae,
    'black_scholes_nn_relative_MAE_price_ge_1':bs_rel_mae_ge1,
    'black_scholes_mc_price':mc_price,
    'black_scholes_mc_runtime_median_s':mc_runtime,
    'black_scholes_mc_runtime_mean_CI95_low_s':mc_runtime_lo,
    'black_scholes_mc_runtime_mean_CI95_high_s':mc_runtime_hi,
    'heston_dataset_size':n_heston,
    'heston_mc_paths_training':HESTON_MC_PATHS_TRAIN,
    'heston_mc_steps_training':HESTON_MC_STEPS_TRAIN,
    'heston_nn_architecture':'9-128-128-64-32-1',
    'heston_nn_loss':'Huber',
    'heston_nn_max_epochs':1000,
    'heston_nn_early_stopping_patience':50,
    'heston_nn_MAE':float(seed_table.loc[best_idx,'MAE']),
    'heston_nn_RMSE':float(seed_table.loc[best_idx,'RMSE']),
    'heston_nn_normalized_MAE':float(seed_table.loc[best_idx,'normalized_MAE']),
    'heston_nn_relative_MAE_price_ge_1':float(seed_table.loc[best_idx,'relative_MAE_price_ge_1']),
    'heston_mc_runtime_median_s':mc_h_time,
    'heston_mc_runtime_mean_CI95_low_s':mc_h_lo,
    'heston_mc_runtime_mean_CI95_high_s':mc_h_hi,
    'heston_nn_runtime_median_s':nn_h_time,
    'heston_nn_runtime_mean_CI95_low_s':nn_h_lo,
    'heston_nn_runtime_mean_CI95_high_s':nn_h_hi,
    'heston_speedup_median_ratio':ratio,
    'heston_speedup_bootstrap_CI95_low':ratio_lo,
    'heston_speedup_bootstrap_CI95_high':ratio_hi,
    'heston_training_time_s':train_cost,
    'heston_breakeven_options':float(breakeven),
    'heston_high_accuracy_MAE':float(hi_mae),
    'heston_high_accuracy_RMSE':float(hi_rmse),
}

from pathlib import Path
OUTPUT_DIR=Path('publication_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
pd.DataFrame([results]).to_csv(OUTPUT_DIR/'publication_benchmark_results.csv',index=False)
print(json.dumps(results,indent=2))
